# Question 5: Kneser-Ney Language Model Smoothing

**Course:** Natural Language Processing (AI357)  
**Institution:** Sardar Vallabhbhai National Institute of Technology (SVNIT), Surat  

---

### Problem Statement:
Consider the following 5 sentences. Don't add any start and end sentence markers:
1. `I like NLP`
2. `I like AI`
3. `You like NLP`
4. `You love AI`
5. `We love NLP`

Use **Kneser-Ney smoothing** to find out:
- $P_{KN}(\text{AI} \mid \text{love})$
- $P_{KN}(\text{NLP} \mid \text{like})$

Use discount **$d = 0.5$** in your computations.  
**Marks:** $[3 + 3 = 6]$


## 1. Mathematical Formulation

For bigrams, the interpolated Kneser-Ney smoothing formula is:

$$P_{KN}(w \mid h) = \frac{\max(c(h, w) - d, 0)}{c(h)} + \lambda(h) \cdot P_{\text{continuation}}(w)$$

where:
- $c(h, w)$ is the count of bigram $(h, w)$
- $c(h)$ is the count of the history word $h$
- $d = 0.5$ is the discount parameter
- $\lambda(h)$ is the backoff normalization weight:
  $$\lambda(h) = \frac{d \cdot |\{w' : c(h, w') > 0\}|}{c(h)}$$
- $P_{\text{continuation}}(w)$ is the continuation probability (number of unique histories preceding $w$, divided by total distinct bigram types):
  $$P_{\text{continuation}}(w) = \frac{|\{h' : c(h', w) > 0\}|}{\text{Total distinct bigram types}}$$


In [1]:
import pandas as pd
from collections import Counter, defaultdict

sentences = [
    ["I", "like", "NLP"],
    ["I", "like", "AI"],
    ["You", "like", "NLP"],
    ["You", "love", "AI"],
    ["We", "love", "NLP"]
]
discount = 0.5

# Step 1: Count bigrams and histories
bigram_counts = Counter()
history_counts = Counter()
history_continuations = defaultdict(set) # {h: set of words following h}
word_preceding_histories = defaultdict(set) # {w: set of words preceding w}

for sentence in sentences:
    for i in range(len(sentence) - 1):
        h, w = sentence[i], sentence[i + 1]
        bigram_counts[(h, w)] += 1
        history_counts[h] += 1
        history_continuations[h].add(w)
        word_preceding_histories[w].add(h)

total_bigram_types = len(bigram_counts)

df_bigrams = pd.DataFrame([
    {"Bigram (h, w)": f"({h}, {w})", "Count c(h, w)": count}
    for (h, w), count in sorted(bigram_counts.items())
])
print(f"Total distinct bigram types (|B|): {total_bigram_types}")
df_bigrams


Total distinct bigram types (|B|): 8


,"Bigram (h, w)","Count c(h, w)"
0,"(I, like)",2
1,"(We, love)",1
2,"(You, like)",1
3,"(You, love)",1
4,"(like, AI)",1
5,"(like, NLP)",2
6,"(love, AI)",1
7,"(love, NLP)",1


In [2]:
# Step 2: Compute Continuation Probabilities
all_words = sorted(list(set(w for pair in bigram_counts for w in pair)))

cont_records = []
p_continuation = {}

for word in all_words:
    preceding_set = word_preceding_histories[word]
    num_preceding = len(preceding_set)
    p_cont = num_preceding / total_bigram_types
    p_continuation[word] = p_cont
    cont_records.append({
        "Word (w)": word,
        "Preceding Histories": str(preceding_set) if num_preceding > 0 else "{}",
        "Distinct Preceding Count": num_preceding,
        "Formula": f"{num_preceding} / {total_bigram_types}",
        "P_continuation(w)": p_cont
    })

df_continuation = pd.DataFrame(cont_records)
print("=== Continuation Probabilities Table ===")
df_continuation


=== Continuation Probabilities Table ===


,Word (w),Preceding Histories,Distinct Preceding Count,Formula,P_continuation(w)
0,AI,"{'like', 'love'}",2,2 / 8,0.25
1,I,{},0,0 / 8,0.00
2,NLP,"{'like', 'love'}",2,2 / 8,0.25
3,We,{},0,0 / 8,0.00
4,You,{},0,0 / 8,0.00
5,like,"{'I', 'You'}",2,2 / 8,0.25
6,love,"{'We', 'You'}",2,2 / 8,0.25


In [3]:
# Step 3: Compute P_KN(AI | love)
h1, w1 = "love", "AI"

c_h1 = history_counts[h1]
c_hw1 = bigram_counts[(h1, w1)]
num_unique_cont_h1 = len(history_continuations[h1])

# Discounted MLE term
term1_1 = max(c_hw1 - discount, 0) / c_h1

# Lambda backoff weight
lambda_h1 = (discount * num_unique_cont_h1) / c_h1

# Backoff continuation term
term2_1 = lambda_h1 * p_continuation[w1]

# Total Kneser-Ney probability
p_kn_1 = term1_1 + term2_1

print(f"=== Computation for P_KN({w1} | {h1}) ===")
print(f"1. Bigram count c(love, AI) = {c_hw1}")
print(f"2. History count c(love)    = {c_h1}")
print(f"3. Distinct words following 'love': {history_continuations[h1]} -> Count = {num_unique_cont_h1}")
print(f"4. Discounted Term: max({c_hw1} - {discount}, 0) / {c_h1} = {term1_1:.4f}  (Fraction: 1/4)")
print(f"5. Lambda weight λ(love): ({discount} * {num_unique_cont_h1}) / {c_h1} = {lambda_h1:.4f}  (Fraction: 1/2)")
print(f"6. Continuation prob P_cont(AI): {p_continuation[w1]:.4f}  (Fraction: 2/8 = 1/4)")
print(f"7. Backoff Term: λ(love) * P_cont(AI) = {lambda_h1:.4f} * {p_continuation[w1]:.4f} = {term2_1:.4f}  (Fraction: 1/8)")
print(f"--> P_KN(AI | love) = {term1_1} + {term2_1} = {p_kn_1:.6f}  (Exact Fraction: 3/8 = 0.375)")


=== Computation for P_KN(AI | love) ===
1. Bigram count c(love, AI) = 1
2. History count c(love)    = 2
3. Distinct words following 'love': {'AI', 'NLP'} -> Count = 2
4. Discounted Term: max(1 - 0.5, 0) / 2 = 0.2500  (Fraction: 1/4)
5. Lambda weight λ(love): (0.5 * 2) / 2 = 0.5000  (Fraction: 1/2)
6. Continuation prob P_cont(AI): 0.2500  (Fraction: 2/8 = 1/4)
7. Backoff Term: λ(love) * P_cont(AI) = 0.5000 * 0.2500 = 0.1250  (Fraction: 1/8)
--> P_KN(AI | love) = 0.25 + 0.125 = 0.375000  (Exact Fraction: 3/8 = 0.375)


In [4]:
# Step 4: Compute P_KN(NLP | like)
h2, w2 = "like", "NLP"

c_h2 = history_counts[h2]
c_hw2 = bigram_counts[(h2, w2)]
num_unique_cont_h2 = len(history_continuations[h2])

# Discounted MLE term
term1_2 = max(c_hw2 - discount, 0) / c_h2

# Lambda backoff weight
lambda_h2 = (discount * num_unique_cont_h2) / c_h2

# Backoff continuation term
term2_2 = lambda_h2 * p_continuation[w2]

# Total Kneser-Ney probability
p_kn_2 = term1_2 + term2_2

print(f"=== Computation for P_KN({w2} | {h2}) ===")
print(f"1. Bigram count c(like, NLP) = {c_hw2}")
print(f"2. History count c(like)     = {c_h2}")
print(f"3. Distinct words following 'like': {history_continuations[h2]} -> Count = {num_unique_cont_h2}")
print(f"4. Discounted Term: max({c_hw2} - {discount}, 0) / {c_h2} = {term1_2:.4f}  (Fraction: 1.5/3 = 1/2)")
print(f"5. Lambda weight λ(like): ({discount} * {num_unique_cont_h2}) / {c_h2} = {lambda_h2:.4f}  (Fraction: 1/3)")
print(f"6. Continuation prob P_cont(NLP): {p_continuation[w2]:.4f}  (Fraction: 2/8 = 1/4)")
print(f"7. Backoff Term: λ(like) * P_cont(NLP) = {lambda_h2:.4f} * {p_continuation[w2]:.4f} = {term2_2:.4f}  (Fraction: 1/12)")
print(f"--> P_KN(NLP | like) = {term1_2:.4f} + {term2_2:.4f} = {p_kn_2:.6f}  (Exact Fraction: 7/12 ≈ 0.583333)")


=== Computation for P_KN(NLP | like) ===
1. Bigram count c(like, NLP) = 2
2. History count c(like)     = 3
3. Distinct words following 'like': {'AI', 'NLP'} -> Count = 2
4. Discounted Term: max(2 - 0.5, 0) / 3 = 0.5000  (Fraction: 1.5/3 = 1/2)
5. Lambda weight λ(like): (0.5 * 2) / 3 = 0.3333  (Fraction: 1/3)
6. Continuation prob P_cont(NLP): 0.2500  (Fraction: 2/8 = 1/4)
7. Backoff Term: λ(like) * P_cont(NLP) = 0.3333 * 0.2500 = 0.0833  (Fraction: 1/12)
--> P_KN(NLP | like) = 0.5000 + 0.0833 = 0.583333  (Exact Fraction: 7/12 ≈ 0.583333)


In [5]:
# Step 5: Summary Table of Results
df_kneser_ney_summary = pd.DataFrame([
    {
        "Probability Query": "P_KN(AI | love)",
        "c(h, w)": c_hw1,
        "c(h)": c_h1,
        "Discounted Term": f"{term1_1:.4f} (1/4)",
        "Lambda λ(h)": f"{lambda_h1:.4f} (1/2)",
        "P_cont(w)": f"{p_continuation[w1]:.4f} (1/4)",
        "Backoff Term": f"{term2_1:.4f} (1/8)",
        "P_KN (Final)": f"{p_kn_1:.6f} (3/8 = 0.375)"
    },
    {
        "Probability Query": "P_KN(NLP | like)",
        "c(h, w)": c_hw2,
        "c(h)": c_h2,
        "Discounted Term": f"{term1_2:.4f} (1/2)",
        "Lambda λ(h)": f"{lambda_h2:.4f} (1/3)",
        "P_cont(w)": f"{p_continuation[w2]:.4f} (1/4)",
        "Backoff Term": f"{term2_2:.4f} (1/12)",
        "P_KN (Final)": f"{p_kn_2:.6f} (7/12 ≈ 0.583333)"
    }
])
print("=== Final Kneser-Ney Summary Table ===")
df_kneser_ney_summary


=== Final Kneser-Ney Summary Table ===


,Probability Query,"c(h, w)",c(h),Discounted Term,Lambda λ(h),P_cont(w),Backoff Term,P_KN (Final)
0,P_KN(AI | love),1,2,0.2500 (1/4),0.5000 (1/2),0.2500 (1/4),0.1250 (1/8),0.375000 (3/8 = 0.375)
1,P_KN(NLP | like),2,3,0.5000 (1/2),0.3333 (1/3),0.2500 (1/4),0.0833 (1/12),0.583333 (7/12 ≈ 0.583333)


## Summary of Answers

1. **$P_{KN}(\text{AI} \mid \text{love})$ [3 Marks]:**
   - Discounted term: $\frac{1 - 0.5}{2} = 0.25 = \frac{1}{4}$
   - Backoff term: $\lambda(\text{love}) \cdot P_{\text{cont}}(\text{AI}) = 0.5 \times 0.25 = 0.125 = \frac{1}{8}$
   - **$P_{KN}(\text{AI} \mid \text{love}) = 0.25 + 0.125 = \mathbf{0.375}$ (or $\frac{3}{8}$)**

2. **$P_{KN}(\text{NLP} \mid \text{like})$ [3 Marks]:**
   - Discounted term: $\frac{2 - 0.5}{3} = 0.5 = \frac{1}{2}$
   - Backoff term: $\lambda(\text{like}) \cdot P_{\text{cont}}(\text{NLP}) = \frac{1}{3} \times 0.25 = \frac{1}{12} \approx 0.083333$
   - **$P_{KN}(\text{NLP} \mid \text{like}) = 0.5 + \frac{1}{12} = \frac{7}{12} \approx \mathbf{0.583333}$**
